In [3]:
import json
import os
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.metrics import make_scorer, mean_absolute_error
from sklearn.model_selection import GridSearchCV, StratifiedKFold, KFold


In [4]:
TRAIN_FEAT_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data/train_features.csv"
TRAIN_TARGET_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/raw_data/customer_clv_train.csv"
BEST_PARAMS_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data/best_params.json"
N_JOBS = -1


In [5]:
required_paths = [TRAIN_FEAT_PATH, TRAIN_TARGET_PATH]
missing_required = [p for p in required_paths if not os.path.exists(p)]
if missing_required:
    raise FileNotFoundError(f"Missing required files: {missing_required}")

train_features = pd.read_csv(TRAIN_FEAT_PATH)
train_target = pd.read_csv(TRAIN_TARGET_PATH)

train_df = train_target.merge(train_features, on="cust_id", how="inner", validate="one_to_one")
feature_cols = [c for c in train_df.columns if c not in ["cust_id", "revenue_2018_2019"]]
X = train_df[feature_cols]
y = train_df["revenue_2018_2019"].to_numpy()
y_bin = (y > 0).astype(int)
y_log = np.log1p(y)

print("X shape:", X.shape)
print("Positive ratio:", y_bin.mean())


X shape: (116591, 248)
Positive ratio: 0.36587729756156134


In [8]:
# 1) Tune classifier (return vs no-return)
clf = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    random_state=42,
    verbose=False,
)

clf_param_grid = {
    "depth": [8],
    "learning_rate": [0.05],
    "iterations": [700, 800],
    "l2_leaf_reg": [3, 5],
}

clf_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
clf_search = GridSearchCV(
    estimator=clf,
    param_grid=clf_param_grid,
    scoring="roc_auc",
    cv=clf_cv,
    n_jobs=N_JOBS,
    verbose=1,
    refit=True,
)
clf_search.fit(X, y_bin)

print("Best classifier AUC:", clf_search.best_score_)
print("Best classifier params:", clf_search.best_params_)


Fitting 3 folds for each of 4 candidates, totalling 12 fits


Best classifier AUC: 0.725903166570315
Best classifier params: {'depth': 8, 'iterations': 700, 'l2_leaf_reg': 5, 'learning_rate': 0.05}


In [9]:
# 2) Tune regressor on log-target
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

reg = CatBoostRegressor(
    loss_function="MAE",
    random_state=42,
    verbose=False,
)

reg_param_grid = {
    "depth": [6, 8],
    "learning_rate": [0.03, 0.05],
    "iterations": [700, 1000],
    "l2_leaf_reg": [3, 5],
}

reg_cv = KFold(n_splits=3, shuffle=True, random_state=42)
reg_search = GridSearchCV(
    estimator=reg,
    param_grid=reg_param_grid,
    scoring=mae_scorer,
    cv=reg_cv,
    n_jobs=N_JOBS,
    verbose=1,
    refit=True,
)
reg_search.fit(X, y_log)

print("Best regressor CV score (negative MAE on log-target):", reg_search.best_score_)
print("Best regressor params:", reg_search.best_params_)


Fitting 3 folds for each of 16 candidates, totalling 48 fits


Best regressor CV score (negative MAE on log-target): -1.4568281265453
Best regressor params: {'depth': 6, 'iterations': 1000, 'l2_leaf_reg': 3, 'learning_rate': 0.03}


In [10]:
best_params = {
    "classifier": {
        "best_score_auc": float(clf_search.best_score_),
        "best_params": clf_search.best_params_,
    },
    "regressor": {
        "best_score_neg_mae_log": float(reg_search.best_score_),
        "best_params": reg_search.best_params_,
    },
}

os.makedirs(os.path.dirname(BEST_PARAMS_PATH), exist_ok=True)
with open(BEST_PARAMS_PATH, "w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2)

print(json.dumps(best_params, indent=2))
print(f"Saved best params to: {BEST_PARAMS_PATH}")


{
  "classifier": {
    "best_score_auc": 0.725903166570315,
    "best_params": {
      "depth": 8,
      "iterations": 700,
      "l2_leaf_reg": 5,
      "learning_rate": 0.05
    }
  },
  "regressor": {
    "best_score_neg_mae_log": -1.4568281265453,
    "best_params": {
      "depth": 6,
      "iterations": 1000,
      "l2_leaf_reg": 3,
      "learning_rate": 0.03
    }
  }
}
Saved best params to: /workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data/best_params.json
